# 📊 Day 1 Homework Grading System: Data Engineering Pipeline
## For Instructors

---

### How to Use
1. Run the Setup cell once only
2. Paste one student's GitHub link → Run grading
3. The result will be **automatically appended** to `day1_scores.csv` and `day1_scores.json`
4. Change the link → Run grading for the next student (no need to restart)

### Scoring Criteria (10 points)

| Step | Full Score | Criteria |
|---------|-----------|-------|
| 1. MD5 Duplicate | 2 | Correctly identify duplicate files + correct hash |
| 2. Chunking | 2 | Correct number of chunks + correct chunk 1 content |
| 3. Embedding + Similarity | 3 | Correct score + reasonable explanation |
| 4. Qdrant Retrieval | 3 | Correct score + high-quality summary |

> **🇹🇭 Thai Text in This Notebook**
>
> Sample data and queries are in Thai because this workshop teaches Thai RAG. English translations are provided as inline comments (`# "translation"`).

## 📦 Setup (Run once)

In [ ]:
%%time
!pip install -q google-genai requests nbformat langchain-text-splitters

import hashlib, os, json, random, re, csv, requests
from datetime import datetime
import nbformat
from google import genai

# Configure Gemini API
try:
    from google.colab import userdata
    GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY')
    print('✅ Loaded API Key from Colab Secrets')
except Exception:
    GOOGLE_API_KEY = input('🔑 กรุณาวาง Gemini API Key: ')  # "🔑 Please paste your Gemini API Key: "

client = genai.Client(api_key=GOOGLE_API_KEY)
MODEL = 'gemini-2.5-pro'

# Score storage files (append mode)
SCORES_CSV = 'day1_scores.csv'
SCORES_JSON = 'day1_scores.json'

# Create CSV file header if it does not exist
if not os.path.exists(SCORES_CSV):
    with open(SCORES_CSV, 'w', encoding='utf-8', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(['Timestamp', 'Student Name', 'Student ID', 'Phone', 'LINE ID',
                         'Step1', 'Step2', 'Step3', 'Step4', 'Total',
                         'AI Suspected', 'Feedback', 'GitHub URL'])
    print(f'📄 Created new file {SCORES_CSV}')
else:
    # Count how many have already been graded
    with open(SCORES_CSV, 'r') as f:
        existing = sum(1 for _ in f) - 1
    print(f'📄 {SCORES_CSV} already contains {existing} students')

# Create JSON file if it does not exist
if not os.path.exists(SCORES_JSON):
    with open(SCORES_JSON, 'w') as f:
        json.dump([], f)

print(f'✅ Setup complete | Model: {MODEL}')

## 🔧 Grading Functions (Run once)

In [ ]:
def generate_expected_answers(student_id):
    """Generate correct answers from the student ID"""
    from langchain_text_splitters import RecursiveCharacterTextSplitter
    
    random.seed(int(hashlib.md5(student_id.encode()).hexdigest()[:8], 16))
    
    all_paragraphs = [
        'การเรียนรู้ของเครื่อง หรือ Machine Learning เป็นสาขาย่อยของปัญญาประดิษฐ์ที่มุ่งเน้นการพัฒนาอัลกอริทึมที่สามารถเรียนรู้จากข้อมูลและปรับปรุงประสิทธิภาพได้โดยอัตโนมัติ โดยไม่จำเป็นต้องเขียนโปรแกรมอย่างชัดเจนสำหรับทุกกรณี',  # "Machine Learning is a subfield of artificial in..."
        'Deep Learning เป็นเทคนิคหนึ่งของ Machine Learning ที่ใช้โครงข่ายประสาทเทียมหลายชั้น Neural Network ในการประมวลผลข้อมูลที่ซับซ้อน เช่น การจดจำภาพ การแปลภาษา และการสร้างข้อความ',  # "Deep Learning is a technique within Machine Lea..."
        'Natural Language Processing หรือ NLP คือสาขาที่เกี่ยวข้องกับการทำให้คอมพิวเตอร์สามารถเข้าใจ ตีความ และสร้างภาษามนุษย์ได้ รวมถึงการวิเคราะห์อารมณ์ การสรุปข้อความ และการตอบคำถาม',  # "Natural Language Processing, or NLP, is the fie..."
        'Retrieval Augmented Generation หรือ RAG เป็นเทคนิคที่รวมการค้นหาข้อมูลเข้ากับการสร้างข้อความของ LLM เพื่อให้ได้คำตอบที่ถูกต้อง ทันสมัย และอ้างอิงแหล่งข้อมูลได้',  # "Retrieval Augmented Generation, or RAG, is a te..."
        'Vector Database เป็นฐานข้อมูลที่ออกแบบมาเพื่อจัดเก็บและค้นหาข้อมูลในรูปแบบ Embedding Vector ช่วยให้สามารถค้นหาข้อมูลที่มีความหมายคล้ายคลึงกันได้อย่างรวดเร็ว',  # "A Vector Database is a database designed to sto..."
        'Text Embedding คือกระบวนการแปลงข้อความให้เป็นชุดตัวเลขหรือ Vector ที่สามารถแสดงความหมายเชิงความหมายของข้อความนั้นได้ ทำให้คอมพิวเตอร์สามารถเปรียบเทียบความคล้ายคลึงระหว่างข้อความได้',  # "Text Embedding is the process of converting tex..."
        'Transformer เป็นสถาปัตยกรรมของ Neural Network ที่ใช้กลไก Attention ในการประมวลผลข้อมูลแบบลำดับ เป็นพื้นฐานของโมเดลภาษาขนาดใหญ่อย่าง GPT BERT และ Gemini',  # "Transformer is a neural network architecture th..."
        'Prompt Engineering คือศาสตร์ของการออกแบบคำสั่งหรือ Prompt ที่ให้กับ LLM เพื่อให้ได้ผลลัพธ์ที่ต้องการ การเขียน Prompt ที่ดีช่วยเพิ่มคุณภาพและความแม่นยำของคำตอบอย่างมาก',  # "Prompt Engineering is the practice of designing..."
        'Fine-tuning คือกระบวนการปรับแต่งโมเดลที่ผ่านการฝึกมาแล้วด้วยข้อมูลเฉพาะทางเพิ่มเติม เพื่อให้โมเดลทำงานได้ดีขึ้นในงานที่เฉพาะเจาะจง เช่น การวิเคราะห์เอกสารทางการแพทย์',  # "Fine-tuning is the process of adapting a pre-tr..."
        'Tokenization คือกระบวนการแบ่งข้อความออกเป็นหน่วยย่อยที่เรียกว่า Token ซึ่งอาจเป็นคำ คำย่อย หรือตัวอักษร สำหรับภาษาไทยการตัดคำมีความซับซ้อนเพราะไม่มีเว้นวรรคระหว่างคำ',  # "Tokenization is the process of splitting text i..."
        'Chunking คือกระบวนการแบ่งเอกสารขนาดยาวออกเป็นส่วนย่อยที่เหมาะสมสำหรับการสร้าง Embedding มีหลายวิธีเช่น Fixed-size Recursive และ Semantic Chunking',  # "Chunking is the process of splitting long docum..."
        'Cosine Similarity เป็นวิธีวัดความคล้ายคลึงระหว่างสอง Vector โดยดูจากมุมระหว่าง Vector ค่า 1 หมายถึงทิศทางเดียวกัน ค่า 0 หมายถึงตั้งฉาก นิยมใช้ในงาน NLP และ Information Retrieval',  # "Cosine Similarity is a method for measuring sim..."
    ]
    
    random.shuffle(all_paragraphs)
    selected = all_paragraphs[:8]
    duplicate_idx = random.randint(0, 5)
    selected.append(selected[duplicate_idx])
    
    hashes = {}
    dup_files = None
    dup_hash = None
    for i, para in enumerate(selected):
        h = hashlib.md5(para.encode()).hexdigest()
        fname = f'doc_{i+1}.txt'
        if h in hashes:
            dup_files = (hashes[h], fname)
            dup_hash = h
        else:
            hashes[h] = fname
    
    unique_texts = []
    seen = set()
    for para in selected:
        h = hashlib.md5(para.encode()).hexdigest()
        if h not in seen:
            unique_texts.append(para)
            seen.add(h)
    
    all_text = '\n\n'.join(unique_texts)
    splitter = RecursiveCharacterTextSplitter(chunk_size=150, chunk_overlap=30)
    chunks = splitter.split_text(all_text)
    
    return {
        'student_id': student_id,
        'dup_files': dup_files,
        'dup_hash': dup_hash,
        'files_after_dedup': len(unique_texts),
        'num_chunks': len(chunks),
        'chunk_1_content': chunks[0] if chunks else '',
        'min_chunk_len': min(len(c) for c in chunks) if chunks else 0,
        'verification_code': hashlib.sha256(f'{student_id}_day1_hw'.encode()).hexdigest()[:12],
    }


def fetch_notebook(github_url):
    """Fetch notebook + student information from GitHub"""
    raw_url = github_url.replace('github.com', 'raw.githubusercontent.com').replace('/blob/', '/')
    print(f'📥 Fetching from: {raw_url}')
    resp = requests.get(raw_url)
    resp.raise_for_status()
    nb = nbformat.reads(resp.text, as_version=4)
    
    info = {'student_name': '', 'student_id': '', 'phone': '', 'line_id': ''}
    full_content = ''
    
    for i, cell in enumerate(nb.cells):
        if cell.cell_type == 'code':
            for key, var in [('student_id','STUDENT_ID'), ('student_name','STUDENT_NAME'),
                             ('phone','PHONE'), ('line_id','LINE_ID')]:
                m = re.search(rf"{var}\s*=\s*['\"]([^'\"]+)['\"]", cell.source)
                if m:
                    info[key] = m.group(1)
        
        ct = cell.cell_type.upper()
        full_content += f'\n\n=== CELL {i} ({ct}) ===\n{cell.source}'
        if hasattr(cell, 'outputs'):
            for out in cell.outputs:
                if hasattr(out, 'text'):
                    full_content += f'\n--- OUTPUT ---\n{out.text}'
                elif hasattr(out, 'data'):
                    for mime, data in out.data.items():
                        if 'text' in mime:
                            txt = data if isinstance(data, str) else '\n'.join(data)
                            full_content += f'\n--- OUTPUT ---\n{txt}'
    
    return info, full_content


def grade_with_gemini(student_id, notebook_content, expected):
    """Use Gemini 2.5 Pro to grade"""
    prompt = f'''คุณเป็นผู้ตรวจการบ้านวิชา AI/RAG สำหรับนักศึกษาปริญญาตรี

# Answer Key
- ไฟล์ซ้ำ: {expected["dup_files"]}, MD5: {expected["dup_hash"]}, หลังลบเหลือ {expected["files_after_dedup"]} ไฟล์  # "dup_files"
- จำนวน chunks: {expected["num_chunks"]}, chunk สั้นสุด: {expected["min_chunk_len"]} ตัวอักษร  # "num_chunks"
- chunk แรก (80 ตัวอักษร): "{expected["chunk_1_content"][:80]}..."
- Verification: {expected["verification_code"]}

# Notebook
{notebook_content[:15000]}

# Criteria
## Step 1: MD5 (2 points) — 1 for correct duplicate files, 0.5 for correct hash, 0.5 for correct file count after deletion
## Step 2: Chunking (2 points) — 1 for correct number of chunks, 0.5 for correct chunk 1 content, 0.5 for correct shortest chunk
## Step 3: Embedding (3 points) — 1 for using multilingual-e5-large, 1 for cosine similarity calculation + score, 1 for self-written reasoning
## Step 4: Qdrant (3 points) — 1 for collection creation + upsert, 1 for Top-3 scores, 1 for high-quality improvement summary

ตอบ JSON:
{{"step1_score":0.0,"step1_feedback":"...","step2_score":0.0,"step2_feedback":"...","step3_score":0.0,"step3_feedback":"...","step4_score":0.0,"step4_feedback":"...","total_score":0.0,"overall_feedback":"...","is_ai_suspected":false,"ai_reason":"..."}}

ให้ feedback เป็นภาษาไทย สั้นกระชับ
ถ้าไม่มี code/output ให้ 0 คะแนน
ถ้าสงสัยใช้ AI เขียนคำอธิบาย ตั้ง is_ai_suspected=true
'''
    resp = client.models.generate_content(
        model=MODEL, contents=prompt,
        config=genai.types.GenerateContentConfig(temperature=0.1, response_mime_type='application/json')
    )
    return json.loads(resp.text)


def append_result(info, grade, github_url):
    """Append grading results to CSV + JSON"""
    ts = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    
    # Append CSV
    with open(SCORES_CSV, 'a', encoding='utf-8', newline='') as f:
        writer = csv.writer(f)
        writer.writerow([
            ts, info['student_name'], info['student_id'], info['phone'], info['line_id'],
            grade['step1_score'], grade['step2_score'], grade['step3_score'], grade['step4_score'],
            grade['total_score'], grade.get('is_ai_suspected', False),
            grade['overall_feedback'], github_url
        ])
    
    # Append JSON
    with open(SCORES_JSON, 'r') as f:
        all_results = json.load(f)
    grade.update({'timestamp': ts, 'github_url': github_url, **info})
    all_results.append(grade)
    with open(SCORES_JSON, 'w', encoding='utf-8') as f:
        json.dump(all_results, f, ensure_ascii=False, indent=2)
    
    # Count total
    print(f'💾 Saved successfully (graded {len(all_results)} students in total)')


print('✅ All functions are ready to use')

---
## ▶️ Grade Homework (one student at a time)

Change `GITHUB_URL` and **rerun the cell below** for each student.
The result will be **automatically appended** to `day1_scores.csv`.

In [ ]:
assert GITHUB_URL, '❌ กรุณาวาง GitHub URL!'  # "❌ Please paste a GitHub URL!"

# 1. Fetch notebook
info, content = fetch_notebook(GITHUB_URL)
print(f'👤 {info["student_name"]} ({info["student_id"]})')
print(f'📱 {info["phone"]} | 💬 {info["line_id"]}')

# 🔍 Check whether this student ID has already been graded
already_graded = False
if os.path.exists(SCORES_JSON):
    with open(SCORES_JSON, 'r') as f:
        prev = json.load(f)
    for p in prev:
        if p.get('student_id') == info['student_id']:
            already_graded = True
            print(f'⚠️  Student ID {info["student_id"]} has already been graded (score: {p.get("total_score", "?")})')
            break

if already_graded:
    overwrite = input('🔄 ต้องการตรวจใหม่และ overwrite คะแนนเดิม? (y/n): ').strip().lower()  # "🔄 Do you want to regrade and overwrite the prev..."
    if overwrite != 'y':
        print('⏭️  Skipping this student')
        raise SystemExit()
    # Remove old data
    prev = [p for p in prev if p.get('student_id') != info['student_id']]
    with open(SCORES_JSON, 'w', encoding='utf-8') as f:
        json.dump(prev, f, ensure_ascii=False, indent=2)
    # Remove from CSV as well
    import csv
    rows = []
    with open(SCORES_CSV, 'r', encoding='utf-8') as f:
        reader = csv.reader(f)
        rows = list(reader)
    with open(SCORES_CSV, 'w', encoding='utf-8', newline='') as f:
        writer = csv.writer(f)
        for row in rows:
            if len(row) > 2 and row[2] == info['student_id']:
                continue
            writer.writerow(row)
    print(f'🗑️  Removed previous score for {info["student_id"]}')

# 2. Generate answer key
expected = generate_expected_answers(info['student_id'])
print(f'🔑 Answer key: dup={expected["dup_files"]}, chunks={expected["num_chunks"]}')

# 3. Grade with Gemini
print(f'🤖 Grading with {MODEL}...')
grade = grade_with_gemini(info['student_id'], content, expected)

# 4. Show results
print(f'\n{"="*60}')
print(f'📊 Grading Result: {info["student_name"]} ({info["student_id"]})')
print(f'{"="*60}')
print(f'  Step 1 (MD5):       {grade["step1_score"]}/2  — {grade["step1_feedback"]}')
print(f'  Step 2 (Chunking):  {grade["step2_score"]}/2  — {grade["step2_feedback"]}')
print(f'  Step 3 (Embedding): {grade["step3_score"]}/3  — {grade["step3_feedback"]}')
print(f'  Step 4 (Qdrant):    {grade["step4_score"]}/3  — {grade["step4_feedback"]}')
print(f'  {"─"*56}')
print(f'  🏆 Total Score: {grade["total_score"]}/10')
print(f'  💬 {grade["overall_feedback"]}')
if grade.get('is_ai_suspected'):
    print(f'  ⚠️  AI suspected: {grade["ai_reason"]}')

# 5. Append result to file
append_result(info, grade, GITHUB_URL)

## 📊 View All Scores

In [ ]:
# Show scores for all graded students
import csv

with open(SCORES_CSV, 'r', encoding='utf-8') as f:
    reader = csv.reader(f)
    header = next(reader)
    rows = list(reader)

print(f'📊 Graded {len(rows)} students so far\n')
print(f'{"#":>3} {"Name":<20} {"ID":<12} {"S1":>4} {"S2":>4} {"S3":>4} {"S4":>4} {"Total":>5} {"AI?":>4}')
print('=' * 72)

for idx, row in enumerate(rows, 1):
    ts, name, sid, phone, line = row[0], row[1], row[2], row[3], row[4]
    s1, s2, s3, s4, total = row[5], row[6], row[7], row[8], row[9]
    ai = '⚠️' if row[10] == 'True' else '✅'
    print(f'{idx:>3} {name:<20} {sid:<12} {s1:>4} {s2:>4} {s3:>4} {s4:>4} {total:>5} {ai:>4}')

# Statistics
if rows:
    scores = [float(r[9]) for r in rows]
    print(f'\n📈 Statistics: min={min(scores):.1f}, max={max(scores):.1f}, avg={sum(scores)/len(scores):.1f}')

## 💾 Download Score File

In [ ]:
# Download CSV (for Google Colab)
try:
    from google.colab import files
    files.download(SCORES_CSV)
    print(f'✅ Downloaded {SCORES_CSV}')
except:
    print(f'📄 Score file is located at: {os.path.abspath(SCORES_CSV)}')